# Black-Litterman + Ledoit-Wolf 协方差收缩模型

**参考论文**: Huixian Zeng (2022), "Research on Black-Litterman Index Enhancement Strategy"  
**论文成果**: 年化收益约36.6%，信息比率(IR) 2.05  
**核心思想**: 将市场均衡收益与投资者主观观点融合，采用Ledoit-Wolf协方差收缩改善估计稳定性

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 策略原理

### Black-Litterman 模型

Black-Litterman模型通过贝叶斯框架将**市场均衡收益**与**投资者观点**融合：

**Step 1: 市场均衡超额收益 (Implied Equilibrium Returns)**

$$\Pi = \delta \Sigma w_{mkt}$$

其中 $\delta$ 为风险厌恶系数，$\Sigma$ 为协方差矩阵，$w_{mkt}$ 为市场权重。

**Step 2: 投资者观点**

$$P \mu = Q + \varepsilon, \quad \varepsilon \sim N(0, \Omega)$$

其中 $P$ 为观点矩阵，$Q$ 为观点收益向量，$\Omega$ 为观点不确定性矩阵。

**Step 3: 后验收益 (Posterior Returns)**

$$\hat{\mu} = [(\tau\Sigma)^{-1} + P^T \Omega^{-1} P]^{-1} [(\tau\Sigma)^{-1}\Pi + P^T \Omega^{-1} Q]$$

### Ledoit-Wolf 协方差收缩

$$\hat{\Sigma}_{LW} = \alpha F + (1-\alpha) S$$

其中 $F$ 为结构化目标（如恒等矩阵的缩放），$S$ 为样本协方差矩阵，$\alpha$ 为最优收缩强度。

### 均值-方差优化

$$\max_w \quad w^T \hat{\mu} - \frac{\delta}{2} w^T \hat{\Sigma} w$$
$$\text{s.t.} \quad \sum w_i = 1, \quad w_i \geq 0$$

In [ ]:
# Cell 3: 动画 - 有效前沿对比（纳入观点前后）
import numpy as np
import pandas as pd
import plotly.graph_objects as go

np.random.seed(42)

def animate_bl_frontier():
    """动画展示BL模型纳入观点前后的有效前沿变化"""
    n_assets = 8
    # 模拟数据
    mu_prior = np.array([0.06, 0.07, 0.12, 0.10, 0.15, 0.18, 0.14, 0.05])
    mu_posterior = np.array([0.08, 0.06, 0.14, 0.11, 0.17, 0.15, 0.13, 0.04])
    cov = np.array([
        [0.04, 0.02, 0.01, 0.01, 0.01, 0.01, 0.01, 0.02],
        [0.02, 0.06, 0.02, 0.01, 0.02, 0.02, 0.02, 0.01],
        [0.01, 0.02, 0.09, 0.02, 0.03, 0.03, 0.02, 0.01],
        [0.01, 0.01, 0.02, 0.05, 0.02, 0.02, 0.01, 0.01],
        [0.01, 0.02, 0.03, 0.02, 0.12, 0.04, 0.03, 0.01],
        [0.01, 0.02, 0.03, 0.02, 0.04, 0.16, 0.04, 0.01],
        [0.01, 0.02, 0.02, 0.01, 0.03, 0.04, 0.10, 0.01],
        [0.02, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.03],
    ])

    def gen_frontier(mu, cov, n=2000):
        rets, risks, sharpes = [], [], []
        for _ in range(n):
            w = np.random.dirichlet(np.ones(n_assets))
            r = w @ mu
            s = np.sqrt(w @ cov @ w)
            rets.append(r)
            risks.append(s)
            sharpes.append((r - 0.025) / s if s > 0 else 0)
        return risks, rets, sharpes

    risks_prior, rets_prior, sh_prior = gen_frontier(mu_prior, cov)
    risks_post, rets_post, sh_post = gen_frontier(mu_posterior, cov)

    frames = []
    batch = 100
    for step in range(batch, 2001, batch):
        frames.append(go.Frame(
            data=[
                go.Scatter(x=risks_prior[:step], y=rets_prior[:step], mode='markers',
                           name='Prior (均衡)', marker=dict(color=sh_prior[:step],
                           colorscale='Blues', size=3, opacity=0.5)),
                go.Scatter(x=risks_post[:step], y=rets_post[:step], mode='markers',
                           name='Posterior (BL)', marker=dict(color=sh_post[:step],
                           colorscale='Reds', size=3, opacity=0.5)),
            ],
            name=f'{step} portfolios'
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title='Black-Litterman: 纳入观点前后有效前沿对比',
        xaxis_title='年化风险 (波动率)', yaxis_title='年化收益',
        height=550, width=850,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 150}, 'transition': {'duration': 80}}]),
        ])],
        sliders=[dict(steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames])],
    )
    return fig

fig = animate_bl_frontier()
fig.show()

In [ ]:
# Cell 4: 导入依赖
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

from shared.data_fetcher import get_etf_daily, get_stock_daily, get_index_daily
from shared.constants import (SECTOR_ETFS, DEFAULT_START, DEFAULT_END,
                               RISK_FREE_RATE, TRADING_DAYS_PER_YEAR, INITIAL_CAPITAL)
from shared.metrics import summary_table
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_full_report)

np.random.seed(42)
print('All imports loaded successfully.')

In [ ]:
# Cell 5: 下载行业ETF数据
etf_names = list(SECTOR_ETFS.keys())
etf_codes = list(SECTOR_ETFS.values())

# 备用个股 (若ETF数据下载失败)
fallback_stocks = {
    '银行': '601398', '券商': '600030', '医药': '600276', '消费': '000858',
    '科技': '002415', '新能源': '300750', '军工': '600893', '地产': '001979',
}

close_dict = {}
for name, code in SECTOR_ETFS.items():
    df = get_etf_daily(code, DEFAULT_START, DEFAULT_END)
    if df is not None and len(df) > 100 and 'close' in df.columns:
        close_dict[name] = df['close']
    else:
        print(f'ETF {name}({code}) 失败, 使用备用个股 {fallback_stocks[name]}')
        df = get_stock_daily(fallback_stocks[name], DEFAULT_START, DEFAULT_END)
        if df is not None and len(df) > 100 and 'close' in df.columns:
            close_dict[name] = df['close']

prices_df = pd.DataFrame(close_dict)
prices_df.sort_index(inplace=True)
prices_df.dropna(how='all', inplace=True)
prices_df.ffill(inplace=True)
prices_df.dropna(inplace=True)

# 基准: 沪深300
bench_df = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
bench_close = bench_df['close'] if len(bench_df) > 0 else None

print(f'数据区间: {prices_df.index[0].date()} ~ {prices_df.index[-1].date()}')
print(f'资产数量: {prices_df.shape[1]}, 交易日: {prices_df.shape[0]}')
prices_df.tail(3)

In [ ]:
# Cell 6: BL模型核心函数 + Ledoit-Wolf协方差

def ledoit_wolf_cov(returns_df):
    """Ledoit-Wolf收缩协方差估计"""
    lw = LedoitWolf().fit(returns_df.dropna())
    return pd.DataFrame(lw.covariance_, index=returns_df.columns, columns=returns_df.columns)


def implied_equilibrium_returns(cov_matrix, market_weights, risk_aversion=2.5):
    """计算市场均衡隐含超额收益: Pi = delta * Sigma * w_mkt"""
    return risk_aversion * cov_matrix.values @ market_weights


def black_litterman_posterior(prior_returns, cov_matrix, P, Q, omega=None, tau=0.05):
    """
    BL后验收益计算
    prior_returns: 先验均衡收益 (N,)
    cov_matrix: 协方差矩阵 (N, N)
    P: 观点矩阵 (K, N)
    Q: 观点收益向量 (K,)
    omega: 观点不确定性 (K, K), 默认用 tau * P @ Sigma @ P^T
    tau: 不确定性标量
    """
    Sigma = cov_matrix.values if hasattr(cov_matrix, 'values') else cov_matrix
    Pi = prior_returns
    n = len(Pi)

    tau_sigma = tau * Sigma
    if omega is None:
        omega = np.diag(np.diag(tau * P @ Sigma @ P.T))

    tau_sigma_inv = np.linalg.inv(tau_sigma)
    omega_inv = np.linalg.inv(omega)

    # 后验收益
    M = np.linalg.inv(tau_sigma_inv + P.T @ omega_inv @ P)
    posterior_mu = M @ (tau_sigma_inv @ Pi + P.T @ omega_inv @ Q)

    # 后验协方差
    posterior_cov = Sigma + M

    return posterior_mu, posterior_cov


def mean_variance_optimize(mu, cov, risk_aversion=2.5):
    """均值-方差优化 (长only, 权重和=1)"""
    n = len(mu)
    def neg_utility(w):
        return -(w @ mu - 0.5 * risk_aversion * w @ cov @ w)
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 0.4)] * n  # 单一资产最多40%
    w0 = np.ones(n) / n
    result = minimize(neg_utility, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    return result.x if result.success else w0


print('BL模型函数定义完成')

In [ ]:
# Cell 7: 滚动窗口BL组合优化

returns_df = prices_df.pct_change().dropna()
asset_names = list(prices_df.columns)
n_assets = len(asset_names)

# 参数设置
lookback = 120           # 回看窗口 (交易日)
rebalance_freq = 20      # 调仓频率 (每月)
risk_aversion = 2.5
tau = 0.05

# 投资者观点设定 (基于行业分析)
# 观点1: 科技板块年化超额收益+5%
# 观点2: 新能源板块年化超额收益+3%
# 观点3: 地产板块年化超额收益-3%

def build_views(asset_names):
    """构建投资者主观观点"""
    n = len(asset_names)
    idx_map = {name: i for i, name in enumerate(asset_names)}

    P = np.zeros((3, n))
    Q = np.zeros(3)

    # 观点1: 科技 > 银行 5% (年化) -> 日频约 0.05/252
    if '科技' in idx_map and '银行' in idx_map:
        P[0, idx_map['科技']] = 1.0
        P[0, idx_map['银行']] = -1.0
        Q[0] = 0.05 / 252

    # 观点2: 新能源绝对收益 +3% 年化
    if '新能源' in idx_map:
        P[1, idx_map['新能源']] = 1.0
        Q[1] = 0.03 / 252

    # 观点3: 地产绝对收益 -3% 年化
    if '地产' in idx_map:
        P[2, idx_map['地产']] = 1.0
        Q[2] = -0.03 / 252

    return P, Q

P, Q = build_views(asset_names)

# 滚动回测
weight_history = []
dates = returns_df.index
rebalance_dates = []

portfolio_returns = pd.Series(0.0, index=dates)
current_weights = np.ones(n_assets) / n_assets

for i in range(lookback, len(dates)):
    # 每日组合收益
    daily_ret = returns_df.iloc[i].values
    portfolio_returns.iloc[i] = current_weights @ daily_ret

    # 调仓日
    if (i - lookback) % rebalance_freq == 0:
        window_returns = returns_df.iloc[i-lookback:i]

        # Ledoit-Wolf协方差
        cov_lw = ledoit_wolf_cov(window_returns)

        # 市场权重 (等权作为近似)
        w_mkt = np.ones(n_assets) / n_assets

        # 隐含均衡收益
        pi = implied_equilibrium_returns(cov_lw, w_mkt, risk_aversion)

        # BL后验
        try:
            post_mu, post_cov = black_litterman_posterior(
                pi, cov_lw, P, Q, tau=tau
            )
            # 均值-方差优化
            current_weights = mean_variance_optimize(post_mu, post_cov, risk_aversion)
        except Exception as e:
            # 如果BL失败, 回退等权
            current_weights = np.ones(n_assets) / n_assets

        # 扣除调仓成本
        if len(rebalance_dates) > 0:
            portfolio_returns.iloc[i] -= 0.002  # 双边交易成本

        rebalance_dates.append(dates[i])
        weight_history.append(current_weights.copy())

# 截取有效区间
portfolio_returns = portfolio_returns.iloc[lookback:]
equity_curve = INITIAL_CAPITAL * (1 + portfolio_returns).cumprod()

print(f'调仓次数: {len(rebalance_dates)}')
print(f'策略区间: {portfolio_returns.index[0].date()} ~ {portfolio_returns.index[-1].date()}')
print(f'终端净值: {equity_curve.iloc[-1]:,.0f}')

In [ ]:
# Cell 8: 回测统计

# 基准收益
bench_returns = None
if bench_close is not None:
    bench_close_aligned = bench_close.reindex(portfolio_returns.index).ffill()
    bench_returns = bench_close_aligned.pct_change().fillna(0)

metrics = summary_table(portfolio_returns, equity_curve, bench_returns)

# 权重分析
weight_df = pd.DataFrame(weight_history, columns=asset_names,
                          index=rebalance_dates[:len(weight_history)])
print('=== 最终权重配置 ===')
last_w = weight_df.iloc[-1]
for name, w in last_w.items():
    print(f'  {name}: {w:.1%}')

print(f'\n=== 绩效指标 ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Cell 9: 可视化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 1) 策略 vs 基准 净值曲线
benchmark_equity = None
if bench_returns is not None:
    benchmark_equity = INITIAL_CAPITAL * (1 + bench_returns).cumprod()
plot_equity_curve(equity_curve, benchmark_equity,
                  title='Black-Litterman + Ledoit-Wolf - 累计收益')

# 2) 回撤图
plot_drawdown(equity_curve, title='Black-Litterman - 回撤')

# 3) 权重堆积图
fig, ax = plt.subplots(figsize=(14, 5))
weight_df.plot.area(ax=ax, alpha=0.75, colormap='Set2')
ax.set_title('BL策略 - 资产权重演变', fontsize=14, fontweight='bold')
ax.set_xlabel('日期')
ax.set_ylabel('权重')
ax.set_ylim(0, 1)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
plt.tight_layout()
plt.show()

# 4) 绩效表
plot_metrics_table(metrics, title='BL + Ledoit-Wolf 绩效指标')

## 结果分析

### 模型特点
1. **Black-Litterman模型**有效融合了市场均衡信息与主观观点，避免了纯均值-方差模型对输入参数过于敏感的问题
2. **Ledoit-Wolf协方差收缩**在有限样本下提供了更稳定的协方差估计，减少了极端权重配置
3. 投资者观点（看好科技/新能源、看空地产）通过贝叶斯框架温和地调整了组合权重

### 实际应用注意
- 观点的设置对结果影响显著，需要有合理的宏观/行业分析支持
- $\tau$ 参数控制先验vs观点的权重，通常取0.01~0.1
- 调仓频率和交易成本在实际中需要权衡
- Ledoit-Wolf相比样本协方差，在资产数/样本数比值较高时优势更明显